# RL-RAG on Colab T4 — end-to-end runner

Trained R0+GRPO checkpoint + `metrics.json` from a cold Colab T4 (16 GB, Linux).

## Before you run
- **Runtime → Change runtime type → T4 GPU → Save.**
- Optional: `HF_TOKEN` in Colab Secrets (left sidebar key icon), avoids HF rate-limit warnings.

## What this does
1. Mount Drive (so results survive disconnects).
2. Clone the repo into Drive.
3. Env vars — HF cache on local disk (`/root/.hfcache`), `VERL_PY` pointing at the 3.12 venv.
4. `setup_colab.sh` — installs Python 3.12 + a venv with torch cu121 + vLLM 0.6.6 + verl at the GRACE pin. Colab's kernel Python 3.13 keeps running the orchestration cells. ~10 min first run, ~1 min on re-runs.
5. CUDA probe.
6. HotpotQA download (tiny + full).
7. BM25 retrieval build (`data/cache/bm25_{train,val}.jsonl`).
8. Baseline eval of un-trained Qwen3-0.6B via HF backend.
9a. Skeleton 1-step verl run (smoke test, must pass before 9b).
9b. Full R0+GRPO training (200 steps, ckpt every 25).
10. Evaluate the final LoRA on val split — writes `metrics.json`.
11. Baseline vs. trained delta + Week-2 Gate PASS/FAIL.

## Why a venv on 3.12?
Colab ships Python 3.13. verl's `pyproject.toml` pins `<3.13,>=3.10` — it refuses to install on 3.13. The venv keeps verl happy without fighting Colab's kernel.

## Session budget
- Steps 1–8: ~15 min cold, <5 min warm.
- Step 9b: 4–8 h on T4 for 200 steps. Fits a free-tier 12 h session. If it disconnects, re-run 9b — `resume_mode: auto` picks up at the last checkpoint.

## 1. Mount Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Clone (or pull) repo into Drive

Change `REPO_URL` to your fork if you've pushed. If you haven't pushed yet, upload the project folder into `/content/drive/MyDrive/rl-rag` by hand and skip the clone — the `elif` branch below handles that.

In [2]:
import os, subprocess, pathlib

REPO_URL   = os.environ.get('REPO_URL',   'https://github.com/elderflamevandle/RL-MSML635-Group-Project.git')
REPO_LOCAL = os.environ.get('REPO_LOCAL', '/content/drive/MyDrive/rl-rag')
BRANCH     = os.environ.get('BRANCH',     'feat/eval-failure-analysis-lj-gate')

p = pathlib.Path(REPO_LOCAL)
if (p / '.git').exists():
    print(f'repo exists at {p}; pulling latest')
    subprocess.run(['git', '-C', str(p), 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', str(p), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(p), 'pull', '--ff-only'], check=True)
elif p.exists() and (p / 'pyproject.toml').exists():
    print(f'repo appears to be at {p} but not a git checkout; using as-is')
else:
    print(f'cloning {REPO_URL} into {p}')
    p.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(p)], check=True)

%cd {REPO_LOCAL}
!git log -1 --oneline

repo exists at /content/drive/MyDrive/rl-rag; pulling latest
/content/drive/MyDrive/rl-rag
49f4191 (HEAD -> feat/eval-failure-analysis-lj-gate, origin/feat/eval-failure-analysis-lj-gate) feat(kaggle): T4 notebook — Kaggle-native paths, secrets, Save-Version guidance


## 3. Persistent env vars (HF cache + PYTHONPATH)

Parking the HF cache on Drive means Qwen3-0.6B (~1.2 GB) only downloads once across all your Colab sessions.

In [3]:
import os
# HF cache on local disk (Drive silently truncates small files).
os.environ['HF_HOME']             = '/root/.hfcache'
os.environ['HF_HUB_CACHE']        = '/root/.hfcache/hub'
os.environ['TRANSFORMERS_CACHE']  = '/root/.hfcache/hub'
os.environ['PYTHONPATH']          = '/content/drive/MyDrive/rl-rag'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
# verl's own pyproject pins Python <3.13; setup_colab.sh installs verl into
# this venv on 3.12. launch_run.sh and the final-eval cell honor VERL_PY.
os.environ['VERL_PY']             = '/content/venv312/bin/python'
os.makedirs(os.environ['HF_HUB_CACHE'], exist_ok=True)
for k in ['HF_HOME','HF_HUB_CACHE','TRANSFORMERS_CACHE','PYTHONPATH','TOKENIZERS_PARALLELISM','VERL_PY']:
    print(f'{k}={os.environ[k]}')

HF_HOME=/content/drive/MyDrive/rl-rag/.hfcache
HF_HUB_CACHE=/content/drive/MyDrive/rl-rag/.hfcache/hub
TRANSFORMERS_CACHE=/content/drive/MyDrive/rl-rag/.hfcache/hub
PYTHONPATH=/content/drive/MyDrive/rl-rag
TOKENIZERS_PARALLELISM=false


## 4. Install deps (~5–8 min, cached after first run)

In [ ]:
!bash scripts/setup_colab.sh

[1/6] python + pip versions
Python 3.13.16
pip 26.2.2 from /usr/local/lib/python3.13/dist-packages/pip (python 3.13)
[2/6] CUDA-capable torch (cu121 wheels, work on T4 sm_75)
ERROR: Could not find a version that satisfies the requirement torchvision==0.20.1 (from versions: 0.1.6, 0.2.0)
ERROR: No matching distribution found for torchvision==0.20.1
cu121 torch install failed; trying cu124
[3/6] HF + retrieval + eval deps
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done


## 5. CUDA sanity

In [ ]:
!nvidia-smi
!python -c "import torch; p=torch.cuda.get_device_properties(0); print(p.name, 'sm_'+str(p.major)+str(p.minor), f'{p.total_memory/1024**3:.1f}GB')"

## 6. HotpotQA data

`hotpot_tiny.jsonl` (50 samples) is used by the baseline eval + skeleton. `hotpot_train.jsonl` + `hotpot_val.jsonl` are the 2000+500 split used for training.

In [ ]:
!python scripts/download_hotpot_tiny.py
!python scripts/download_hotpot_full.py
!ls -lh data/raw/

## 7. Build retrieval cache (BM25 only — CPU, ~1 min)

This writes `data/cache/bm25_{train,val}.jsonl` with the `prompt` column verl expects. The T4 config points at these files. If you later want dense retrieval, drop `--no-dense` and re-run — then swap the `data.train_files` paths in `configs/r0_grpo_t4.yaml` to `dense_train.jsonl` / `dense_val.jsonl`.

In [ ]:
!python scripts/build_retrieval.py --no-dense
!ls -lh data/cache/
!echo '--- first row of bm25_train.jsonl ---'
!python -c "import json; row=json.loads(open('data/cache/bm25_train.jsonl').readline()); print('keys:', list(row)); assert 'prompt' in row, 'prompt field missing — verl will fail'; print('prompt starts:', row['prompt'][:200])"

## 8. Baseline eval (pre-RL metrics)

Evaluates the un-trained Qwen3-0.6B via the HF backend on the tiny set. This is the "before" number for the final results table. ~3–5 min on T4.

In [ ]:
!python -m src.eval.evaluate \n    --checkpoint Qwen/Qwen3-0.6B \n    --split data/cache/bm25_val.jsonl \n    --out results/baseline_t4/metrics.json \n    --backend hf --device cuda --max-new-tokens 256 \n    --max-samples 50
!echo '--- baseline metrics ---'
!cat results/baseline_t4/metrics.json

## 9a. Smoke test — 1-step verl run with the T4 skeleton config

This is the critical validation step: if verl + vLLM + the config schema and reward-fn path all agree, this exits cleanly. If it fails, read the verl error — field names in `configs/skeleton_t4.yaml` almost certainly need tweaking vs. the installed verl. Edit the YAML, re-run the cell. **Do not proceed to step 9b until this passes.**

In [ ]:
!bash scripts/launch_run.sh configs/skeleton_t4.yaml

## 9b. Full R0+GRPO training (200 steps, ~4–8 h on T4)

Writes checkpoints every 25 steps to `results/<run_id>/checkpoints/`. If Colab disconnects, re-run just this cell — `resume_mode: auto` picks up from the last checkpoint.

**Live tail:** open a second cell with `!tail -f results/*/train.log` after this one starts, or watch the streaming output below.

In [ ]:
!bash scripts/launch_run.sh configs/r0_grpo_t4.yaml

## 10. Evaluate the trained checkpoint → `metrics.json`

Finds the latest checkpoint from step 9b, evaluates it on the val split, writes `metrics.json` + `predictions.jsonl` alongside the training log. These are the files the Week-2 Gate + comparison table consume.

Set `GROQ_API_KEY` via `%env GROQ_API_KEY=...` and add `--judge` below to include LLM-as-Judge scores.

In [ ]:
import glob, os, subprocess

run_dirs = sorted(glob.glob('results/r0_grpo_qwen3-0.6b_s0_*'))
assert run_dirs, 'No r0_grpo_* run dir found under results/ — did step 9b complete?'
run_dir = run_dirs[-1]
print('run_dir:', run_dir)

ckpts = sorted(glob.glob(f'{run_dir}/checkpoints/global_step_*'))
assert ckpts, f'No checkpoints under {run_dir}/checkpoints/ — training did not save'
ckpt = ckpts[-1]
print('checkpoint:', ckpt)

out = f'{run_dir}/metrics.json'
PY = os.environ.get('VERL_PY', '/content/venv312/bin/python')
cmd = [
    PY, '-m', 'src.eval.evaluate',
    '--checkpoint', ckpt,
    '--split', 'data/cache/bm25_val.jsonl',
    '--out', out,
    '--backend', 'vllm',
    '--device', 'cuda',
    '--max-new-tokens', '384',
    '--bertscore',
]
print('running:', ' '.join(cmd))
subprocess.run(cmd, check=True)
print('---')
print(open(out).read())

## 11. Compare baseline vs. trained

Quick side-by-side so you can see whether 200 steps of R0+GRPO actually moved Balanced Accuracy.

In [ ]:
import json, glob

baseline = json.load(open('results/baseline_t4/metrics.json'))
run_dir  = sorted(glob.glob('results/r0_grpo_qwen3-0.6b_s0_*'))[-1]
trained  = json.load(open(f'{run_dir}/metrics.json'))

keys = ['balanced_accuracy', 'em_answerable', 'abstention_unanswerable']
print(f'{"metric":<28} {"baseline":>10} {"trained":>10} {"delta":>10}')
for k in keys:
    b = baseline.get(k, float('nan'))
    t = trained.get(k, float('nan'))
    print(f'{k:<28} {b:>10.3f} {t:>10.3f} {t-b:>+10.3f}')

print()
print('Week-2 Gate: trained balanced_accuracy must be in [0.71, 0.91].')
ba = trained['balanced_accuracy']
status = 'PASS' if 0.71 <= ba <= 0.91 else 'FAIL'
print(f'  balanced_accuracy = {ba:.3f}  →  {status}')

## 12. Commit results back (optional)

If you cloned from a writable remote, push the metrics + log so your teammates see them. Otherwise `results/` lives on Drive and you can download by hand.

In [ ]:
import subprocess
subprocess.run(['git', 'config', 'user.email', 'colab@rl-rag'], check=True)
subprocess.run(['git', 'config', 'user.name',  'Colab Runner'], check=True)
subprocess.run(['git', 'add', 'results/'], check=True)
status = subprocess.run(['git', 'status', '--short'], capture_output=True, text=True).stdout
print(status)
if status.strip():
    subprocess.run(['git', 'commit', '-m', 'results: Colab T4 R0+GRPO run + metrics'], check=True)
    print('Commit created. Run `!git push` to push.')
else:
    print('Nothing to commit.')